In [ ]:
%cd ../..
import os
from PIL import Image
import numpy as np
import torch
from tqdm import tqdm
from omegaconf import OmegaConf

from dinov2.inference import generate_embeddings, build_model, view_volume, crop_volume, load_dicom

In [ ]:
sample_path = "/scratch/VM/radio-foundation/datasets-nodicom/CCCII/CP/2429/2890"
img = load_volume(sample_path)
view_volume(img, (5.0, 1.0, 1.0))

print(img.shape, img.min(), img.max())

In [ ]:
config_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/config.yaml"
checkpoint_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/eval/training_99999/teacher_checkpoint.pth"

device = torch.device("cuda")

config = OmegaConf.load(config_path)
model, autocast_ctx = build_model(checkpoint_path, config, img_size=504, device=device)

In [ ]:
data_path = "/scratch/VM/radio-foundation/datasets-nodicom/CCCII"
output_path = "/scratch/VM/radio-foundation/cache/embeddings/CCCCII"

In [ ]:
import traceback

data_kwargs = dict(
    fmean = -573.8,
    fstd = 461.3,
    channels = 10,
    img_size = 504,
    patch_size = 14,
    device="cuda",
    block_size=64,
    autocast_ctx=autocast_ctx,
)
class_names = ["CP", "NCP", "Normal"]
for c in class_names:
    print(c)
    base_path = os.path.join(data_path, c)
    os.makedirs(os.path.join(output_path, c), exist_ok=True)
    for id_i in tqdm(os.listdir(base_path)):
        id_i_path = os.path.join(base_path, id_i)
        for id_j in os.listdir(id_i_path):
            scan_path = os.path.join(id_i_path, id_j)
            new_id = f"{id_i:04}_{id_j:04}"

            p_output_dir = os.path.join(output_path, c, f"{new_id}.pth")
            if os.path.exists(p_output_dir):
                continue

            try:
                img = load_volume(folder_path=scan_path)

                collated_features = generate_embeddings(
                    img,
                    model=model,
                    **data_kwargs # type: ignore
                )

                output = {"cls": collated_features["cls"]}

                torch.save(output, p_output_dir)
            except Exception:
                print(new_id, scan_path)
                print(traceback.format_exc())

